# Klasifikasi Citra Penyakit Kulit Multi-Kelas dengan Transfer Learning (MobileNetV2)

Notebook ini disusun untuk **reimplementasi-terkontrol** dan ekstensi paper 5-kelas menjadi multi-kelas menggunakan dataset Kaggle.  
Fokus utama: **MobileNetV2** (wajib), **Xception** (opsional pembanding).

> Catatan etika: hasil model ini adalah **pengenalan awal berbasis citra**, **bukan diagnosis medis**.

## 1) Setup Environment Google Colab
**Tujuan cell:** menyiapkan dependensi dan koneksi Kaggle API jika dataset belum tersedia lokal.

**Alasan teknis:** Colab bersifat ephemeral, jadi setup perlu eksplisit agar eksperimen dapat direproduksi.

In [ ]:
# Jika diperlukan, aktifkan GPU: Runtime > Change runtime type > GPU

import os
import random
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2, Xception
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.xception import preprocess_input as xception_preprocess

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

## 2) (Opsional) Download Dataset dari Kaggle
**Tujuan cell:** mengunduh dataset jika belum ada di Drive/Colab.

**Alasan teknis:** memudahkan eksekusi end-to-end dalam satu notebook.

In [ ]:
# OPSIONAL: Jalankan jika dataset belum tersedia.
# Pastikan file kaggle.json sudah diunggah ke /content/

# !mkdir -p ~/.kaggle
# !cp /content/kaggle.json ~/.kaggle/kaggle.json
# !chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d ismailpromus/skin-diseases-image-dataset -p /content
# !unzip -q /content/skin-diseases-image-dataset.zip -d /content/data_skin

print("Silakan set DATA_ROOT ke folder dataset hasil ekstraksi.")

## 3) Set Path Dataset dan Audit Struktur Folder (WAJIB sebelum training)
**Tujuan cell:** memeriksa struktur dataset otomatis, mendeteksi folder kelas, dan validasi file gambar.

**Alasan teknis:** mencegah error pipeline dan memastikan eksperimen dimulai dari data yang benar.

In [ ]:
# Ubah sesuai lokasi dataset Anda di Colab/Drive
DATA_ROOT = Path('/content/data_skin')

# Contoh alternatif:
# DATA_ROOT = Path('/content/drive/MyDrive/skin-diseases-image-dataset')

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def find_class_root(data_root: Path):
    """
    Mencari direktori yang berisi subfolder kelas gambar.
    Jika data_root langsung berisi subfolder kelas, gunakan data_root.
    Jika ada 1 level pembungkus, telusuri otomatis.
    """
    if not data_root.exists():
        raise FileNotFoundError(f"Path tidak ditemukan: {data_root}")

    def is_class_dir(p: Path):
        if not p.is_dir():
            return False
        subdirs = [d for d in p.iterdir() if d.is_dir()]
        if len(subdirs) == 0:
            return False
        # dianggap root kelas jika mayoritas subfolder berisi gambar
        hit = 0
        for sd in subdirs:
            imgs = [f for f in sd.rglob('*') if f.suffix.lower() in IMG_EXTS]
            if len(imgs) > 0:
                hit += 1
        return hit >= max(1, int(0.6 * len(subdirs)))

    if is_class_dir(data_root):
        return data_root

    candidates = [d for d in data_root.iterdir() if d.is_dir() and is_class_dir(d)]
    if len(candidates) == 1:
        return candidates[0]
    elif len(candidates) > 1:
        print("Ditemukan beberapa kandidat root kelas:")
        for c in candidates:
            print(" -", c)
        raise ValueError("Tentukan DATA_ROOT lebih spesifik ke salah satu kandidat di atas.")
    else:
        raise ValueError("Tidak menemukan struktur folder kelas yang valid.")

CLASS_ROOT = find_class_root(DATA_ROOT)
print("CLASS_ROOT:", CLASS_ROOT)

## 4) Audit Dataset: Daftar Kelas + Jumlah Gambar per Kelas + Visualisasi Distribusi
**Tujuan cell:** menampilkan kelas yang terdeteksi dan menghitung jumlah gambar per kelas dalam tabel & plot.

**Alasan teknis:** memahami ketidakseimbangan data sejak awal sebelum split/training.

In [ ]:
# Kumpulkan data file
records = []
class_dirs = sorted([d for d in CLASS_ROOT.iterdir() if d.is_dir()])

for cdir in class_dirs:
    class_name = cdir.name
    files = [f for f in cdir.rglob('*') if f.suffix.lower() in IMG_EXTS]
    for fp in files:
        records.append({'filepath': str(fp), 'label': class_name})

df = pd.DataFrame(records)
if df.empty:
    raise ValueError("Tidak ada gambar ditemukan. Periksa path/ekstensi file.")

class_counts = df['label'].value_counts().sort_index()
class_table = class_counts.reset_index()
class_table.columns = ['class_name', 'num_images']

print(f"Jumlah total gambar: {len(df)}")
print(f"Jumlah kelas terdeteksi: {class_table.shape[0]}")
print("Daftar kelas:")
for i, c in enumerate(class_table['class_name'].tolist(), 1):
    print(f"{i:02d}. {c}")

display(class_table)

plt.figure(figsize=(12,5))
sns.barplot(data=class_table, x='class_name', y='num_images', palette='viridis')
plt.title('Distribusi Jumlah Gambar per Kelas')
plt.xlabel('Kelas')
plt.ylabel('Jumlah Gambar')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 5) Split Data Train/Validation/Test (Stratified bila memungkinkan)
**Tujuan cell:** membuat split 70:15:15 (default) dengan stratifikasi label.

**Alasan teknis:** menjaga proporsi kelas antar split dan mencegah data leakage.

In [ ]:
SPLIT_RATIO = (0.7, 0.15, 0.15)  # train, val, test
assert abs(sum(SPLIT_RATIO)-1.0) < 1e-6

train_ratio, val_ratio, test_ratio = SPLIT_RATIO

# Coba stratified split; jika gagal (kelas terlalu sedikit), fallback non-stratified
stratify_labels = df['label']

try:
    df_train, df_temp = train_test_split(
        df,
        test_size=(1-train_ratio),
        random_state=SEED,
        stratify=stratify_labels
    )
    val_relative = val_ratio / (val_ratio + test_ratio)
    df_val, df_test = train_test_split(
        df_temp,
        test_size=(1-val_relative),
        random_state=SEED,
        stratify=df_temp['label']
    )
    used_stratify = True
except ValueError as e:
    print("Peringatan stratified split gagal:", e)
    print("Fallback ke split acak non-stratified.")
    df_train, df_temp = train_test_split(df, test_size=(1-train_ratio), random_state=SEED)
    val_relative = val_ratio / (val_ratio + test_ratio)
    df_val, df_test = train_test_split(df_temp, test_size=(1-val_relative), random_state=SEED)
    used_stratify = False

print("Stratified split digunakan:", used_stratify)
print("Train:", len(df_train), "| Val:", len(df_val), "| Test:", len(df_test))

# Audit distribusi split
split_df = pd.concat([
    df_train.assign(split='train'),
    df_val.assign(split='val'),
    df_test.assign(split='test')
])

display(pd.crosstab(split_df['label'], split_df['split']))

## 6) Data Pipeline TensorFlow (tanpa leakage)
**Tujuan cell:** membuat `tf.data.Dataset` untuk train/val/test. Augmentasi hanya untuk train.

**Alasan teknis:** test set harus mencerminkan data nyata (tanpa augmentasi) agar evaluasi adil.

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

class_names = sorted(df['label'].unique().tolist())
class_to_index = {c:i for i,c in enumerate(class_names)}
num_classes = len(class_names)

print("Class mapping:")
for k,v in class_to_index.items():
    print(f"{k:25s} -> {v}")

def df_to_arrays(df_part):
    x = df_part['filepath'].values
    y = df_part['label'].map(class_to_index).values
    return x, y

x_train, y_train = df_to_arrays(df_train)
x_val, y_val = df_to_arrays(df_val)
x_test, y_test = df_to_arrays(df_test)

def load_and_resize(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)
    return img, label

train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train)).shuffle(len(x_train), seed=SEED)
val_ds = tf.data.Dataset.from_tensor_slices((x_val, y_val))
test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test))

train_ds = train_ds.map(load_and_resize, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
val_ds = val_ds.map(load_and_resize, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
test_ds = test_ds.map(load_and_resize, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

## 7) Bangun Model MobileNetV2 (Transfer Learning)
**Tujuan cell:** membuat model dengan base MobileNetV2 pretrained ImageNet, dibekukan pada tahap awal.

**Alasan teknis:** fitur umum dari ImageNet biasanya membantu saat data medis terbatas.

In [ ]:
# Layer augmentasi (hanya dipakai saat training=True)
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
], name='data_augmentation')

base_model = MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # tahap 1: freeze

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = mobilenet_preprocess(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation='relu')(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model = keras.Model(inputs, outputs, name='MobileNetV2_skin_multiclass')

# Adam dipilih karena umumnya stabil untuk fine-tuning bertahap
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 8) Callback Training
**Tujuan cell:** menyiapkan penyimpanan model terbaik dan kontrol overfitting.

In [ ]:
ckpt_path = '/content/best_mobilenetv2_skin.keras'

callbacks = [
    keras.callbacks.ModelCheckpoint(
        ckpt_path,
        monitor='val_accuracy',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

## 9) Training Tahap 1 (Classifier Head, Base Frozen)
**Tujuan cell:** melatih classifier head terlebih dahulu.

In [ ]:
EPOCHS_STAGE1 = 15

history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_STAGE1,
    callbacks=callbacks
)

## 10) (Opsional) Fine-Tuning Tahap 2
**Tujuan cell:** membuka sebagian layer akhir base model dengan learning rate kecil.

**Alasan teknis:** setelah head stabil, fine-tuning dapat meningkatkan adaptasi fitur domain kulit.

In [ ]:
DO_FINE_TUNE = True

if DO_FINE_TUNE:
    base_model.trainable = True

    # Bekukan layer awal, buka layer akhir saja
    fine_tune_at = int(len(base_model.layers) * 0.8)
    for layer in base_model.layers[:fine_tune_at]:
        layer.trainable = False

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-5),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    EPOCHS_STAGE2 = 10
    history_stage2 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_STAGE2,
        callbacks=callbacks
    )
else:
    history_stage2 = None

## 11) Plot Kurva Training vs Validation (Accuracy dan Loss)
**Tujuan cell:** memvisualisasikan dinamika belajar model.

In [ ]:
def merge_history(h1, h2=None):
    hist = {k:list(v) for k,v in h1.history.items()}
    if h2 is not None:
        for k,v in h2.history.items():
            hist.setdefault(k,[])
            hist[k].extend(v)
    return hist

hist = merge_history(history_stage1, history_stage2)

plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.plot(hist['accuracy'], label='train_acc')
plt.plot(hist['val_accuracy'], label='val_acc')
plt.title('Training vs Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1,2,2)
plt.plot(hist['loss'], label='train_loss')
plt.plot(hist['val_loss'], label='val_loss')
plt.title('Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

## 12) Evaluasi pada Test Set
**Tujuan cell:** menghitung accuracy, precision, recall, F1-score, classification report, dan confusion matrix.

In [ ]:
# Load model terbaik dari checkpoint
best_model = keras.models.load_model(ckpt_path)

y_prob = best_model.predict(test_ds)
y_pred = np.argmax(y_prob, axis=1)

y_true = y_test  # dari split sebelumnya

acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)

print(f"Test Accuracy : {acc:.4f}")
print(f"Test Precision: {prec:.4f}")
print(f"Test Recall   : {rec:.4f}")
print(f"Test F1-score : {f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - Test Set')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 13) Fungsi Prediksi 1 Gambar Baru (Top-3)
**Tujuan cell:** inferensi praktis per-gambar dengan probabilitas top-3.

In [ ]:
def predict_single_image(image_path, model, class_names, img_size=(224,224)):
    img_raw = tf.io.read_file(image_path)
    img = tf.image.decode_image(img_raw, channels=3, expand_animations=False)
    img = tf.image.resize(img, img_size)
    img = tf.cast(img, tf.float32)

    # Tampilkan gambar
    plt.figure(figsize=(4,4))
    plt.imshow(tf.cast(img, tf.uint8).numpy())
    plt.title('Input Image')
    plt.axis('off')
    plt.show()

    # Prediksi
    x = tf.expand_dims(img, axis=0)
    probs = model.predict(x, verbose=0)[0]

    top3_idx = np.argsort(probs)[-3:][::-1]
    print('Top-3 Prediksi:')
    for rank, idx in enumerate(top3_idx, start=1):
        print(f"{rank}. {class_names[idx]} -> {probs[idx]*100:.2f}%")

# Contoh penggunaan:
# sample_path = x_test[0]
# predict_single_image(sample_path, best_model, class_names, IMG_SIZE)

## 14) (Opsional) Model Pembanding Xception
Gunakan hanya jika resource cukup.

In [ ]:
RUN_XCEPTION = False

if RUN_XCEPTION:
    x_base = Xception(include_top=False, weights='imagenet', input_shape=IMG_SIZE + (3,))
    x_base.trainable = False

    inp = keras.Input(shape=IMG_SIZE + (3,))
    z = data_augmentation(inp)
    z = xception_preprocess(z)
    z = x_base(z, training=False)
    z = layers.GlobalAveragePooling2D()(z)
    z = layers.Dropout(0.3)(z)
    out = layers.Dense(num_classes, activation='softmax')(z)
    xception_model = keras.Model(inp, out, name='Xception_skin_multiclass')

    xception_model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    xception_model.summary()
    # Lanjutkan training sama seperti MobileNetV2 jika diperlukan.

## 15) Template Interpretasi Hasil
Gunakan template ini setelah mendapatkan metrik.

- Jika kelas A sering salah menjadi kelas B, kemungkinan karena **kemiripan visual lesi** atau **ketidakseimbangan data**.  
- Jika train accuracy jauh di atas val accuracy, indikasi **overfitting** (perlu regularisasi/augmentasi tambahan).  
- Jika recall kelas minoritas rendah, pertimbangkan **class weighting**, **focal loss**, atau **penambahan data kelas minoritas**.

## 16) Kesimpulan Eksperimen (Template)
1. **Performa model pada dataset multi-kelas (mis. 10 kelas):** tulis ringkasan accuracy, precision, recall, F1 test.  
2. **Kelas paling sulit diklasifikasikan:** identifikasi dari confusion matrix dan F1 per kelas terendah.  
3. **Kompleksitas vs paper 5 kelas:** jelaskan bahwa perluasan kelas meningkatkan ambiguitas visual dan batas keputusan model.  
4. **Keterbatasan eksperimen:** ukuran data per kelas tidak seimbang, kualitas citra beragam, keterbatasan epoch/compute Colab.  
5. **Saran pengembangan:** fine-tuning lebih sistematis, balancing strategy, uji arsitektur lain (EfficientNet/ConvNeXt), dan validasi eksternal.

> Penegasan: eksperimen ini untuk **klasifikasi citra penyakit kulit** (pengenalan awal), bukan diagnosis medis.